<a href="https://colab.research.google.com/github/Teddy-2004/prob-bayesian-gradient-descent/blob/main/Bayesian.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Dataset Setup:

Due to GitHub's file size limit of 25MB, the IMDB Dataset.csv file dataset is not hosted directly in this repository.

* **Source:** [Download from Kaggle](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews)
* File Name: `IMDB Dataset.csv`

To run the Bayesian Probability analysis in this notebook:
1. Download the CSV file from the Kaggle link above.
2. Place the file in the same root directory as this notebook.
3. The "from scratch" implementation will then be able to load and process the 50,000 reviews as documented below.

In [ ]:
import pandas as pd
df = pd.read_csv('IMDB Dataset.csv')
print("--- Class Balance ---")
print(df['sentiment'].value_counts())
print("\n--- Data Preview ---")
print(df.head())

--- Class Balance ---
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

--- Data Preview ---
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


# Part 2: Bayesian Probability Implementation

### Keyword Rationale
To evaluate the sensitivity of our Bayesian model we selected the following keywords to analyze their impact on sentiment or their expected emotional intensity:

* High-Intensity Positive: "brilliant" and "amazing" were chosen because they are typically reserved for high-quality cinema.

* High-Intensity Negative: "worst", "waste", and "awful" represent clear, unambiguous dissatisfaction.


* Neutral/Ambiguous: "talented" was selected as a "control" word. We hypothesized that it might appear in both positive and negative reviews (e.g., "a talented actor in a bad movie"), testing if the model can distinguish between specific praise and overall sentiment.


In [ ]:
def get_bayesian_probabilities(keyword, df):
  total_reviews = len(df)
  total_pos = len(df[df['sentiment'] == 'positive'])
  has_keyword = df['review'].str.contains(keyword, case=False, na=False)
  count_keyword = has_keyword.sum()
  count_keyword_and_pos = (has_keyword & (df['sentiment'] == 'positive')).sum()
  # initial probability P(Positive) based on dataset balance
  prior_pos = total_pos / total_reviews

  # P(Keyword | Positive) - how often keyword appears in positive reviews
  likelihood = count_keyword_and_pos / total_pos if total_pos > 0 else 0

  # P(keyword) - total occurence of the word in all reviews
  marginal = count_keyword / total_reviews if total_reviews > 0 else 0

  # P(Positive | keyword) using Bayes' Theorem formula
  posterior = (likelihood * prior_pos) / marginal if marginal > 0 else 0

  return {
      "keyword": keyword,
      "Prior P(Pos)": round(prior_pos, 4),
      "Likelihood P(key|Pos)": round(likelihood, 4),
      "Marginal P(key)": round(marginal, 4),
      "Posterior P(Pos|key)": round(posterior, 4)
  }

keywords = ["brilliant", "talented", "amazing", "worst", "waste", "boring"]
results = [get_bayesian_probabilities(k, df) for k in keywords]
results_df = pd.DataFrame(results)
print(results_df)
results_df_sorted = results_df.sort_values(by="Posterior P(Pos|key)", ascending=False)
results_df_sorted

     keyword  Prior P(Pos)  Likelihood P(key|Pos)  Marginal P(key)  \
0  brilliant           0.5                 0.0755           0.0489   
1   talented           0.5                 0.0229           0.0223   
2    amazing           0.5                 0.0740           0.0496   
3      worst           0.5                 0.0164           0.0887   
4      waste           0.5                 0.0145           0.0731   
5     boring           0.5                 0.0247           0.0623   

   Posterior P(Pos|key)  
0                0.7711  
1                0.5125  
2                0.7463  
3                0.0927  
4                0.0993  
5                0.1983  


,keyword,Prior P(Pos),Likelihood P(key|Pos),Marginal P(key),Posterior P(Pos|key)
0,brilliant,0.5,0.0755,0.0489,0.7711
2,amazing,0.5,0.0740,0.0496,0.7463
1,talented,0.5,0.0229,0.0223,0.5125
5,boring,0.5,0.0247,0.0623,0.1983
4,waste,0.5,0.0145,0.0731,0.0993
3,worst,0.5,0.0164,0.0887,0.0927


### Analysis of Bayesian Updates

Our results demonstrate that the model successfully updated our Prior probability (0.5) into a more certain Posterior probability based on the evidence:


* Strong Evidence: The word "worst" provided the strongest negative evidence, shifting our belief from a 50% chance of a positive review down to only 9.27%.

* Positive Shift: Conversely, "brilliant" increased our confidence in a positive sentiment to 77.11%.


*  Model Sensitivity: The word "talented" resulted in a posterior of 51.25%, confirming our hypothesis that it is a neutral predictor. This proves the model is sensitive enough to realize that not all "positive-sounding" words are strong indicators of a positive review.